# Layered Bayesian network analysis

This notebook runs a complete analysis: it learns a network whose arcs are
constrained by an expert ordering of variable layers, measures how stable each
arc is under resampling, ranks variables by how much they tell you about each
outcome, and reads outcome probabilities for participant profiles you specify.

**You do not need to edit any code in this notebook.** Every choice the
analysis makes is read from `spec.yml`: which variables exist, which layer each
belongs to, the order of the layers, the outcomes, the discretisation, the
number of resamples, the profiles. To analyse your own cohort you edit that
file and re-run this notebook from top to bottom.

There are exactly two settings in the next cell, and they are the only lines
here you are expected to change.

### Before you start

1. **An analysis-ready table.** One row per participant, one column per
   variable, no missing values. This package does not preprocess: deriving
   outcomes, applying censoring, canonicalising dropout categories and imputing
   are decisions specific to your cohort, and they belong in your own script
   that runs before this one.
2. **`spec.yml`**, describing that table. Start from the template and replace
   the layers and variables with yours.
3. **`config.yml`** (optional), holding the paths on your machine. Keeping
   paths out of `spec.yml` is what makes the spec publishable alongside a
   manuscript.

Run the cells in order. If one fails, `docs/troubleshooting.md` lists the
messages you are most likely to see and what each one means.

## 1. Setup

In [ ]:
# The two settings you change.
SPEC_PATH = "spec.yml"      # your analysis specification
USE_DEMO_DATA = True        # set to False to use your own data instead

# Which network from spec.yml the sections below use. It must be one of the
# `variants` names in the spec.
VARIANT = "joint"

In [ ]:
from pathlib import Path
import warnings

import pandas as pd

from vcibayes.analysis import Analysis, read_table
from vcibayes.spec import load_spec

# pyAgrum and scikit-learn emit deprecation notices that are not actionable
# from here and would otherwise bury the output.
warnings.filterwarnings("ignore")

pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 200)

## 2. Load the specification and the data

The spec is validated as it is read. If something is wrong with it you get an
error naming the exact key, before any time is spent on model fitting.

In [ ]:
spec = load_spec(SPEC_PATH)

if USE_DEMO_DATA:
    # A small simulated cohort so this notebook runs before your own data is
    # ready. Its variable names are placeholders and mean nothing.
    from vcibayes.demo import make_demo_cohort
    df = make_demo_cohort(n=1200, seed=0)
    OUTPUT_DIR = Path("outputs")
else:
    # Paths come from config.yml when there is one, so that this notebook and
    # the spec stay free of machine-specific locations.
    config_path = Path("config.yml")
    if config_path.exists():
        import yaml
        config = yaml.safe_load(config_path.read_text()) or {}
        data_dir = Path(config.get("output_dir", ".")).expanduser()
        OUTPUT_DIR = Path(config.get("project_root", ".")).expanduser() / "outputs"
    else:
        data_dir = Path(".")
        OUTPUT_DIR = Path("outputs")
    df = read_table(data_dir / spec.data_path)

print(f"{len(df)} rows, {len(df.columns)} columns")
df.head()

Building the `Analysis` object checks the spec against the data and reports any
disagreement between them. Two kinds are worth acting on:

- **Missing from the data**: the spec declares a variable that is not a column.
  It cannot be modelled. Either the spec has a typo or the preprocessing did
  not produce that column.
- **Not in any layer**: the data has a column the spec does not mention. It is
  set aside, so the network contains exactly what the spec declares. Add it to
  a layer if it belongs in the analysis.

In [ ]:
study = Analysis(spec, df)
print(study.summary())

## 3. Discretisation

Structure learning here works on discrete variables, so every continuous
variable is cut into bins. **Read this table before interpreting anything
else**: every result below is conditional on these bin edges, and a variable
binned too coarsely can look unrelated to an outcome when it is not.

If an ordinal score has been cut up when it should have been left alone, raise
`discretisation.threshold` in the spec. If the bins are too coarse, raise
`n_bins`, at the cost of splitting the data thinner.

In [ ]:
study.bins(VARIANT)

## 4. Learn the network

The layer order in the spec is the constraint: an arc may run from a layer to
itself or to a layer further down the list, never back up. Two further rules
apply automatically:

- no arcs between outcomes, so one endpoint is never treated as a cause of
  another;
- arcs into the dropout layer only from outcomes, and every outcome is
  connected to it after learning. That is how the analysis represents
  informative dropout rather than assuming it away.

Node colour follows the layer order.

In [ ]:
bn = study.network(VARIANT)
print(f"{len(bn.names())} variables, {len(bn.arcs())} arcs")

study.draw(VARIANT)

## 5. How stable is each arc?

A single learned network overstates what the data support: run it again on a
slightly different sample and some arcs move. Each dataset below is a resample
of the participants, with the structure and the parameters relearned from
scratch, and the frequency is the fraction of resamples containing that arc.

This is the slow step. Its cost is set by `bootstrap.n` in the spec, so keep
that small while you are setting the analysis up.

There is no principled cutoff for "stable". Report the frequency rather than
only the arcs that clear some threshold.

In [ ]:
edges = study.stable_edges(VARIANT)
edges.head(25)

`In network` says whether the arc is in the network learned from the full data.
That is a different question from how often it survives resampling, and the two
disagree in both directions: a frequent arc can be absent from the single
network, and an arc in the single network can be rare under resampling.

The figure below is the one to report. Arc width is the mutual information
carried by the arc, and arc colour and the `f=NN%` label are the bootstrap
frequency, so a wide pale arc is informative but unstable.

In [ ]:
study.draw(
    VARIANT,
    stability=True,
    save_path=OUTPUT_DIR / f"network_{VARIANT}.pdf",
)

## 6. Which variables matter for each outcome

Two rankings per outcome, answering different questions.

**MI** is how much a variable tells you about the outcome on its own. It counts
information arriving through any path, including one that runs entirely through
other variables.

**CMI** conditions on the outcome's parents in the network: what the variable
adds *beyond* the variables already adjacent to the outcome. A variable with
high MI and near-zero CMI is explained away by its neighbours, which is
informative rather than disappointing. It is empty for the parents themselves,
since those form the conditioning set.

Neither is an effect estimate, and neither is causal.

In [ ]:
info = study.information(VARIANT)
info.round(4)

## 7. Layer ablation

A network learned with a layer excluded, compared against the same network with
it included. This is how to answer "does this group of variables change the
picture?" — a question that comes up whenever part of the data was collected
for only some participants, or is expensive enough that a reviewer will ask
whether it earned its place.

Which layers are excluded is a property of each variant in the spec, so the
comparison is between two variants rather than two code paths. Adjust the names
below to the variants your spec defines.

In [ ]:
ABLATION_VARIANT = "with_markers"   # a variant with a different `exclude_layers`

outcome_vars = [
    v for layer in spec.layers if layer.role == "outcome" for v in layer.variables
]

rows = []
for name in (VARIANT, ABLATION_VARIANT):
    network = study.network(name)
    excluded = list(spec.variant(name).exclude_layers) or ["(none)"]
    for outcome in outcome_vars:
        if outcome not in network.names():
            continue
        parents = sorted(
            network.variable(p).name()
            for p in network.parents(network.idFromName(outcome))
        )
        rows.append({
            "Variant": name,
            "Excluded layers": ", ".join(excluded),
            "Outcome": outcome,
            "Parents of the outcome": ", ".join(parents) or "(none)",
        })

pd.DataFrame(rows)

If the outcome's parents are unchanged, the excluded layer sits upstream and
does not reach the outcomes directly, and leaving it out costs nothing. If they
change, that layer carries information the rest of the network does not.

## 8. Outcome risk for specific profiles

Fix a set of variables at chosen values and read the outcome probabilities from
the network, with a bootstrap interval on each. The profiles come from
`analyses.scenarios.profiles` in the spec.

Values may be written the way you would say them. A number lands in whichever
bin contains it, and the cell prints what each value was snapped to, so you can
confirm that "60" meant the bin you expected. Anything that could not be
matched is reported by name rather than dropped silently.

In [ ]:
risks = study.scenarios(VARIANT)
risks.round(3)

## 9. Turning one knob

Section 8 gives one number per fixed profile. This section instead holds a
profile fixed and moves a single variable through each of its states, showing
how the outcome probabilities respond. The variable, the fixed profile and the
outcomes to plot all come from `analyses.knob_sweep` in the spec.

**What to hold fixed.** The knob acts on the outcomes through the variables
between them. Fixing one of those blocks the pathway and the result goes flat,
not because the knob is irrelevant but because its effect has been conditioned
away. Hold constant only variables *upstream* of the knob. The cell prints a
warning if the profile fixes something downstream of it.

**Reading the figure.** One panel per outcome, one line per outcome state, and
the shaded band is the 95% bootstrap interval. A line that rises or falls shows
the outcome is sensitive to the knob; the width of the band shows how well
determined that is. A flat line with a narrow band is a real finding.

These are conditional probabilities under the learned distribution. They
describe participants who differ in the knob, not the effect of intervening to
change it in one participant.

In [ ]:
sweep, sweep_meta = study.knob_sweep(VARIANT)
study.plot_sweep(sweep, sweep_meta)
sweep.round(3)

## 10. Save the results

Written to `OUTPUT_DIR` rather than to whatever directory the kernel happens to
have started in, so re-running from elsewhere does not scatter files.

The `.bifxml` file is the network itself: structure and conditional probability
tables. It can be reopened later, or in other Bayesian network software,
without repeating the analysis.

In [ ]:
import pyagrum as gum

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

gum.saveBN(bn, str(OUTPUT_DIR / f"network_{VARIANT}.bifxml"))
edges.to_csv(OUTPUT_DIR / f"edge_stability_{VARIANT}.csv", index=False)
info.to_csv(OUTPUT_DIR / f"information_{VARIANT}.csv", index=False)
risks.to_csv(OUTPUT_DIR / f"scenario_risks_{VARIANT}.csv", index=False)
sweep.to_csv(OUTPUT_DIR / f"knob_sweep_{VARIANT}.csv", index=False)

for path in sorted(OUTPUT_DIR.iterdir()):
    print(path)

## Adapting this to your own cohort

Everything below happens in `spec.yml`. None of it needs a change here.

| You want to | Edit in `spec.yml` |
| --- | --- |
| Use your own variables | `layers` — replace them, keeping causal order |
| Change what causes what | Reorder the `layers` list. That order is the constraint |
| Use a different outcome | Move it into the layer marked `role: outcome` |
| Model dropout | Put it in a layer marked `role: selection` |
| Drop dropout modelling | Delete the `role: selection` layer |
| Add or remove a network | `variants` |
| Leave a layer out | `exclude_layers` on that variant |
| Change the binning | `discretisation` |
| Make it run faster | Lower `bootstrap.n` |
| Change the profiles | `analyses.scenarios.profiles` |
| Sweep a different variable | `analyses.knob_sweep.knob` |

Then set `USE_DEMO_DATA = False` in section 1, point `data.path` at your table,
and run the notebook from the top.

Two things worth recording in a manuscript, because results cannot be
reproduced without them: `model.seed`, and the discretisation table from
section 3.